# Deep learning on text

## Load modules from repo

In [1]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [2]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [3]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_text import preprocess_features
from src.preprocessing.pipelines.deep_learning import load_preprocessors
from src.models.on_text.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

2025-10-06 16:24:22.082025: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-06 16:24:22.128305: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-06 16:24:23.142462: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_text)
importlib.reload(src.models.on_text.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

<module 'src.models.on_text_and_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_text_and_images/deep_learning.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')
full_X_train=X_train
full_y_train=y_train

In [9]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [10]:
version=1
artifacts_folder=Path(f'artifacts/on_text/deep_learning/v{version}')
log_file_path=Path(f'artifacts/on_images/deep_learning/v1') / 'experiments.parquet'  # Same as for images
preprocessors_folder=Path(f'artifacts/on_images/deep_learning/v1')  # Same as for images

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = True  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

BATCH_SIZE = 32

RANDOM_SEED = 42

# load_model=True
load_model=False

## Preprocessing (TODO)

In [11]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [12]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [13]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024700
min       0.008980
25%       0.018696
50%       0.031797
75%       0.054541
max       0.122185
Name: proportion, dtype: float64

In [14]:
y_train.value_counts().describe()

count     27.000000
mean     251.592593
std      167.786325
min       61.000000
25%      127.000000
50%      216.000000
75%      370.500000
max      830.000000
Name: count, dtype: float64

In [15]:
print(X_train.shape)

(6793, 31)


In [16]:
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target'],artifacts_folder=preprocessors_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib


In [17]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, full_X_train=full_X_train, full_y_train=full_y_train, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights)
preprocessors |= new_preprocessors

I0000 00:00:1759760664.488469   17184 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1225 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [18]:
new_preprocessors

{'text_vectorizer': <TextVectorization name=text_vectorization, built=False>}

In [19]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [20]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False)

## Model

In [21]:
from tensorflow import keras

### Load or create model

In [22]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    arch_version = last_experiment.get('arch_version', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    arch_version = last_experiment.get('arch_version', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model = define_model(text_vectorizer=preprocessors['text_vectorizer'], num_classes=27)

Création d'un nouveau modèle.


In [23]:
if not load_model:
    arch_version = int(input(f"arch_version? (last: {arch_version})"))

In [24]:
arch_version

10

### Summary

In [41]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_input (InputLayer)         │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 310)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_embedding (Embedding)      │ (None, 310, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 306, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 27)             │         1,755 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,652,059 (10.12 MB)

 Trainable params: 2,652,059 (10.12 MB)

 Non-trainable params: 0 (0.00 B)

## Callbacks

### ModelCheckpoint

In [26]:
# # Pick an available filename to save a model.
# subversion=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{arch_version}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     arch_version+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{arch_version}.h5')
# new_location_for_saving_model


In [27]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [28]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [29]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [30]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_accuracy', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='max',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [31]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [32]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [33]:
import datetime
tensor_board_folder = artifacts_folder / "tensorboard_logs"
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder / timestamp,
    histogram_freq=1 # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [34]:
import math
typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793

In [35]:
max_epochs=3

# Calculate expected duration
available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
available_minutes

7

In [36]:
# # Pick max_epochs based on your available time
# available_minutes=30

# max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
# max_epochs

### compilation and callbacks

In [37]:
learning_rate=0.001

In [38]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [39]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [40]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=7, max_epochs=3, champion_path=None ?

In [ ]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=total_epochs_trained, callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 1/4


2025-10-06 13:04:05.207523: I external/local_xla/xla/service/service.cc:163] XLA service 0x74b5f8001920 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-06 13:04:05.207572: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-06 13:04:05.532030: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-06 13:04:06.867344: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-06 13:04:07.748758: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-06 13:04:07.

  46/2123 ━━━━━━━━━━━━━━━━━━━━ 6:17 182ms/step - accuracy: 0.5179 - loss: 1.8724

## Evaluation

In [ ]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

source venv/bin/activate
tensorboard --logdir '/home/val/Documents/Dev/DataScientest/Rakuten/artifacts/on_images/deep_learning/v1/tensorboard_logs'


In [ ]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 1,
 'minutes_per_epoch': 8.565354001522063}

In [ ]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [ ]:
#Takes 1m40
y_pred = model.predict(test_ds)

531/531 ━━━━━━━━━━━━━━━━━━━━ 72s 126ms/step


array([[5.03017625e-04, 9.96413408e-04, 2.24974472e-03, ...,
        1.37626985e-02, 3.07596754e-04, 4.64344514e-04],
       [1.94570306e-03, 5.69425290e-03, 5.19817229e-03, ...,
        4.01135832e-02, 1.51008205e-03, 8.56529921e-03],
       [1.37812676e-04, 7.38932751e-03, 1.86753348e-02, ...,
        1.66873947e-01, 3.35931283e-04, 7.14386115e-04],
       ...,
       [1.87164575e-01, 2.55546370e-03, 1.34333255e-04, ...,
        4.94825363e-05, 7.38174856e-01, 9.09753377e-04],
       [6.52999559e-04, 1.41401717e-03, 2.88966112e-03, ...,
        3.13488096e-02, 5.83381450e-04, 8.57704494e-04],
       [5.43505931e-03, 5.20436326e-03, 1.68522149e-02, ...,
        1.06006429e-01, 4.18848917e-03, 5.76869585e-04]],
      shape=(16984, 27), dtype=float32)

In [ ]:
from sklearn import metrics

In [ ]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [ ]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1180,1280,1281,1300,1301,1302,1320,1560,1920,1940,2060,2220,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,,,,,
10,347,8,0,0,7,5,0,3,4,3,0,0,0,0,1,3,2,0,119,61,1,12,0,1,0,45,1
40,46,185,6,3,24,15,0,5,9,35,0,2,1,3,3,1,15,0,73,15,18,7,0,5,16,10,5
50,5,8,62,11,35,3,0,18,4,56,0,6,8,7,0,1,21,0,2,9,14,14,0,44,8,0,0
60,1,10,8,99,0,1,0,3,4,11,0,0,0,0,0,0,5,0,3,0,8,7,0,3,0,0,3
1140,16,8,0,0,419,6,0,22,6,6,0,2,2,0,2,1,10,0,10,2,2,8,0,10,0,0,2
1160,19,9,0,0,20,683,0,0,2,0,0,0,0,0,1,0,0,0,39,10,7,1,0,0,0,0,0
1180,14,4,1,0,57,6,1,7,4,3,0,0,3,1,1,2,5,0,15,10,1,5,0,9,0,2,2
1280,4,3,3,1,143,3,0,388,30,161,2,14,5,15,13,3,73,0,5,10,1,22,1,36,28,3,7
1281,16,19,0,0,36,20,0,67,70,13,1,7,0,3,1,12,32,0,17,27,2,26,0,21,4,9,11


In [ ]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.5658471612426285, 0.0)

In [ ]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [ ]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

,precision,recall,f1-score,support
10,0.438131,0.556982,0.490459,623.000000
40,0.583596,0.368526,0.451770,502.000000
50,0.568807,0.184524,0.278652,336.000000
60,0.831933,0.596386,0.694737,166.000000
1140,0.432851,0.784644,0.557923,534.000000
1160,0.881290,0.863464,0.872286,791.000000
1180,1.000000,0.006536,0.012987,153.000000
1280,0.440909,0.398357,0.418554,974.000000
1281,0.395480,0.169082,0.236887,414.000000
1300,0.610853,0.780971,0.685515,1009.000000


In [ ]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.622432,0.505878,0.502795,629.037037
std,0.168268,0.278884,0.222021,420.759339
min,0.334197,0.006536,0.012987,153.000000
25%,0.471607,0.296960,0.355040,310.000000
50%,0.620339,0.556982,0.492632,534.000000
75%,0.710000,0.781790,0.676668,953.500000
max,1.000000,0.908046,0.872286,2042.000000


In [ ]:
# positive correlation between support and another measure can suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,-0.040267,0.060419,-0.028859
recall,-0.040267,1.000000,0.952164,0.080412
f1-score,0.060419,0.952164,1.000000,0.073780
support,-0.028859,0.080412,0.073780,1.000000


In [ ]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.5658471612426285

In [ ]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 1,
 'minutes_per_epoch': 8.565354001522063,
 'weighted_avg_f1_score': 0.5658471612426285,
 'min_f1_score': np.float64(0.012987012987012988),
 'std_f1_score': np.float64(0.22202115720211937)}

## Update tracker

In [ ]:
tracker['comment']="Full training set. Set callbacks to val_accuracy instead of val_loss."
tracker['comment']

NameError: name 'tracker' is not defined

In [ ]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Sauvegarde du modèle.")
    keep_candidate=True

    best_epoch_in_session_idx = np.argmax(model_history.history['val_accuracy'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_accuracy = model_history.history['val_accuracy'][best_epoch_in_session_idx]
    tracker['val_accuracy'] = best_val_accuracy

    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_arch-{arch_version}_epoch_index-{best_epoch_global:02d}_val_accuracy-{best_val_accuracy:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_filename)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print("Effacement de l'ancien modèle de la même arch_version {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le modèle chargé.")
    keep_candidate=False


Nouveau champion ! Sauvegarde du modèle.
best_model_sv-9_epoch_index-00_val_accuracy-0.5903_f1-0.5658.keras


In [ ]:
tracker['epoch_index'] = best_epoch_global
tracker['total_epochs'] = total_epochs_trained + best_epoch_global + 1
tracker['modality'] = 'text'

In [ ]:
to_track=['arch_version','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate','timestamp','modality']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model, base_model)

In [ ]:
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 1,
 'minutes_per_epoch': 8.565354001522063,
 'weighted_avg_f1_score': 0.5658471612426285,
 'min_f1_score': np.float64(0.012987012987012988),
 'std_f1_score': np.float64(0.22202115720211937),
 'comment': 'Full training set.',
 'val_accuracy': 0.5902613997459412,
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-9_epoch_index-00_val_accuracy-0.5903_f1-0.5658.keras',
 'epoch_index': np.int64(0),
 'total_epochs': np.int64(1),
 'subversion': 9,
 'max_epochs': 1,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'timestamp': '20251006-124449',
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16}}

In [ ]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [ ]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [ ]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [ ]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, loaded_model=load_model, log_file_path=log_file_path)

Log pour l'expérience subversion 9 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet .


## Show tracking logs

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/on_text/deep_learning/v1/experiments.parquet'

In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
